# BindCraft input and notebook audit

## tl;dr

This companion notebook audits the supplied `BindCraft-0823.ipynb` and the eight peptide PDB inputs without modifying them. The supplied notebook contains 12 code cells but no saved execution counts or outputs. Two negative-target structures are incomplete relative to their deposited biological sequences: GIP is 30/42 residues and oxyntomodulin is 26/37 residues. These gaps block a strong computational-selectivity claim.

## Context & Methods

The audit checks notebook execution evidence, PDB chain/residue/atom coverage, missing backbone atoms, explicit hydrogen atoms, zero-occupancy atoms, and C-alpha RMSD across positive GLP-1 conformers. Expected GIP and oxyntomodulin lengths come from the corresponding RCSB entries (7DTY and 7LLY).

### Key Assumptions

- A missing notebook output is treated as absence of execution evidence, not proof that the notebook was never run elsewhere.
- PDB coverage is measured against the deposited peptide length, not only against resolved coordinates.
- RMSD is descriptive of the supplied coordinates and does not establish solution-state populations.

In [ ]:
from pathlib import Path
from collections import OrderedDict
import json
import numpy as np
import pandas as pd

project_root = Path.cwd()
if not (project_root / 'BindCraft-0823.ipynb').exists():
    if (project_root.parent / 'BindCraft-0823.ipynb').exists():
        project_root = project_root.parent
    else:
        project_root = Path('$PROJECT_ROOT/bindcraft')
panel_dir = project_root / 'glp1_target_panel'
notebook_path = project_root / 'BindCraft-0823.ipynb'
assert panel_dir.is_dir() and notebook_path.is_file()


## Data

The source grain is one extracted peptide-chain PDB per file. The panel contains three intact GLP-1(7-36) positive conformers, two GLP-1(9-36) truncated negatives, and one file each for GIP, glucagon, and oxyntomodulin.

In [ ]:
expected_lengths = {
    'GIP_7DTY.pdb': 42,
    'GLP1_7_36_1D0R_model1.pdb': 30,
    'GLP1_7_36_1D0R_model10.pdb': 30,
    'GLP1_7_36_6X18.pdb': 30,
    'GLP1_9_36_1D0R_model1.pdb': 28,
    'GLP1_9_36_6X18.pdb': 28,
    'Glucagon_6LMK.pdb': 29,
    'Oxyntomodulin_7LLY.pdb': 37,
}

def profile_pdb(path):
    residues = OrderedDict()
    atom_count = hydrogen_count = zero_occupancy_count = 0
    chains = set()
    for line in path.read_text().splitlines():
        if not line.startswith('ATOM'):
            continue
        atom_count += 1
        chain = line[21].strip() or '_'
        chains.add(chain)
        residue_key = (chain, line[22:26].strip(), line[26].strip())
        residues.setdefault(residue_key, set()).add(line[12:16].strip())
        element = line[76:78].strip().upper()
        if element == 'H' or line[12:16].strip().startswith('H'):
            hydrogen_count += 1
        if float(line[54:60]) == 0:
            zero_occupancy_count += 1
    missing_backbone = sum(not {'N', 'CA', 'C', 'O'}.issubset(atoms) for atoms in residues.values())
    expected = expected_lengths[path.name]
    return {
        'file': path.name, 'chains': ','.join(sorted(chains)),
        'observed_residues': len(residues), 'expected_residues': expected,
        'coverage': len(residues) / expected, 'atoms': atom_count,
        'hydrogen_atoms': hydrogen_count, 'zero_occupancy_atoms': zero_occupancy_count,
        'residues_missing_backbone': missing_backbone,
    }

pdb_profile = pd.DataFrame(profile_pdb(path) for path in sorted(panel_dir.glob('*.pdb')))
pdb_profile


## Results

The next checks quantify the supplied positive-conformer spread, notebook execution evidence, and a static property of the custom N-terminal selection score.

In [ ]:
def ca_coordinates(path):
    rows = []
    for line in path.read_text().splitlines():
        if line.startswith('ATOM') and line[12:16].strip() == 'CA':
            rows.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])
    return np.asarray(rows, dtype=float)

def kabsch_rmsd(first, second):
    first = first - first.mean(axis=0)
    second = second - second.mean(axis=0)
    u, _, v = np.linalg.svd(first.T @ second)
    rotation = u @ np.diag([1, 1, np.linalg.det(u @ v)]) @ v
    return float(np.sqrt(np.mean(np.sum((first @ rotation - second) ** 2, axis=1))))

positive_files = [
    'GLP1_7_36_6X18.pdb',
    'GLP1_7_36_1D0R_model1.pdb',
    'GLP1_7_36_1D0R_model10.pdb',
]
rmsd_rows = []
for index, first_name in enumerate(positive_files):
    for second_name in positive_files[index + 1:]:
        first = ca_coordinates(panel_dir / first_name)
        second = ca_coordinates(panel_dir / second_name)
        rmsd_rows.append({'first': first_name, 'second': second_name, 'CA_RMSD_A': kabsch_rmsd(first, second)})
pd.DataFrame(rmsd_rows)


In [ ]:
source_notebook = json.loads(notebook_path.read_text())
code_cells = [cell for cell in source_notebook['cells'] if cell['cell_type'] == 'code']
execution_evidence = pd.DataFrame([{
    'total_cells': len(source_notebook['cells']),
    'code_cells': len(code_cells),
    'executed_code_cells': sum(cell.get('execution_count') is not None for cell in code_cells),
    'cells_with_outputs': sum(bool(cell.get('outputs')) for cell in code_cells),
    'total_saved_outputs': sum(len(cell.get('outputs', [])) for cell in code_cells),
}])
execution_evidence


In [ ]:
residue_count = 2
minimum_n_terminal_contacts = 2
contact_weight = 0.025
max_contact_count = residue_count
passing_contact_counts = list(range(minimum_n_terminal_contacts, max_contact_count + 1))
pd.DataFrame({
    'passing_contact_count': passing_contact_counts,
    'selection_score_contact_term': [contact_weight * count for count in passing_contact_counts],
})


## Takeaways

- The notebook is a thoughtful prototype, but the supplied artifact contains no saved evidence of execution and no BindCraft result CSVs, designed PDBs, logs, or experimental measurements.
- GIP and oxyntomodulin are incomplete negatives. In particular, the oxyntomodulin file omits its distinctive C-terminal extension, so it is not a valid full oxyntomodulin counter-screen.
- The two-residue N-terminal contact gate has a maximum value of two and requires two to pass; therefore every passing design receives exactly the same contact contribution of 0.05 to `Selection_Score`.
- The supplied evidence supports a code-and-input review only. It does not support a claim that selective binders were generated or experimentally validated.